# Model: Co-DINO (Collaborative DINO)

End-to-end collaborative detection transformer with hybrid assignment training.

**Architecture**: Swin-T backbone → ChannelMapper neck → CoDINO decoder (one-to-one + one-to-many collaborative assignment)

**Framework**: MMDetection 3.x + MMEngine

**Training**: AdamW + step LR, 36 epochs, best checkpoint saved by val mAP@50

**Key advantage**: NMS-free inference, better recall on dense/overlapping litter objects vs anchor-based models

## Installation

Run once to install MMDetection 3.x and its dependencies.

In [ ]:
import subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-U', 'openmim', '-q'])
for pkg in ['mmengine', 'mmcv>=2.0.0,<2.2.0', 'mmdet>=3.3.0']:
    subprocess.check_call([sys.executable, '-m', 'mim', 'install', pkg, '-q'])
print('Installation complete.')

## GPU Verification & Imports

In [ ]:
import torch
import os, glob, json, numpy as np, cv2, time
from collections import defaultdict
from tqdm import tqdm
import matplotlib.pyplot as plt

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

import mmdet
print(f'MMDetection: {mmdet.__version__}')

## Paths & Constants

In [ ]:
BASE_DIR    = 'C:/Users/User/Desktop/Ipynb'
DATASET_DIR = os.path.join(BASE_DIR, 'archive', 'dataset_v2')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'runs', 'co_dino')
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_CLASSES    = 9      # mmdet excludes background from class count
CONF_THRESHOLD = 0.5

CLASS_NAMES = ('Bottle', 'Cigarette', 'Foam', 'Glass', 'Metal',
               'Other', 'Paper', 'Plastic', 'Unlabeled')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Output: {OUTPUT_DIR}')
print(f'Device: {DEVICE}')

## Register TACO Dataset

Inherit from `CocoDataset` since TACO uses COCO-format annotations.

In [ ]:
from mmdet.registry import DATASETS
from mmdet.datasets import CocoDataset

@DATASETS.register_module(force=True)
class TACODataset(CocoDataset):
    METAINFO = dict(
        classes=('Bottle', 'Cigarette', 'Foam', 'Glass', 'Metal',
                 'Other', 'Paper', 'Plastic', 'Unlabeled'),
        palette=[
            (220, 20, 60), (119, 11, 32), (0, 0, 142), (0, 0, 230),
            (106, 0, 228), (0, 60, 100), (0, 80, 100), (0, 0, 70), (0, 0, 192)
        ]
    )

print('TACODataset registered.')

## Data Pipelines

In [ ]:
img_scale = (1024, 1024)

train_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(type='RandomResize', scale=img_scale, ratio_range=(0.5, 2.0), keep_ratio=True),
    dict(type='RandomCrop', crop_size=img_scale, allow_negative_crop=True),
    dict(type='RandomFlip', prob=0.5),
    dict(type='PhotoMetricDistortion'),
    dict(type='PackDetInputs'),
]

test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='Resize', scale=img_scale, keep_ratio=True),
    dict(type='LoadAnnotations', with_bbox=True),
    dict(
        type='PackDetInputs',
        meta_keys=('img_id', 'img_path', 'ori_shape', 'img_shape', 'scale_factor'),
    ),
]

print('Pipelines defined.')

## Co-DINO Model Config

CoDETR with Swin-T backbone. Components:
- **Backbone**: SwinTransformer-Tiny (pretrained on ImageNet)
- **Neck**: ChannelMapper (projects to 256-ch, 4 FPN scales)
- **Query head**: CoDINOHead (6-layer encoder + 6-layer decoder, 300 queries)
- **Aux heads**: RPNHead + Shared2FCBBoxHead + ATSSHead (collaborative training only)

In [ ]:
model_cfg = dict(
    type='CoDETR',
    num_queries=300,
    with_box_refine=True,
    as_two_stage=True,
    eval_module='detr',
    data_preprocessor=dict(
        type='DetDataPreprocessor',
        mean=[123.675, 116.28, 103.53],
        std=[58.395, 57.12, 57.375],
        bgr_to_rgb=True,
        pad_size_divisor=1,
    ),
    backbone=dict(
        type='SwinTransformer',
        embed_dims=96,
        depths=[2, 2, 6, 2],
        num_heads=[3, 6, 12, 24],
        window_size=7,
        mlp_ratio=4,
        qkv_bias=True,
        drop_rate=0.,
        attn_drop_rate=0.,
        drop_path_rate=0.2,
        patch_norm=True,
        out_indices=(1, 2, 3),
        with_cp=False,
        convert_weights=True,
        init_cfg=dict(
            type='Pretrained',
            checkpoint='https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth',
        ),
    ),
    neck=dict(
        type='ChannelMapper',
        in_channels=[192, 384, 768],
        kernel_size=1,
        out_channels=256,
        act_cfg=None,
        bias=True,
        norm_cfg=dict(type='GN', num_groups=32),
        num_outs=4,
    ),
    query_head=dict(
        type='CoDINOHead',
        num_query=300,
        num_classes=NUM_CLASSES,
        in_channels=2048,
        sync_cls_avg_factor=True,
        with_box_refine=True,
        as_two_stage=True,
        mixed_selection=True,
        dn_cfg=dict(
            label_noise_scale=0.5,
            box_noise_scale=1.0,
            group_cfg=dict(dynamic=True, num_groups=None, num_dn_queries=100),
        ),
        transformer=dict(
            type='CoDinoTransformer',
            with_coord_feat=False,
            num_co_heads=2,
            num_feature_levels=4,
            encoder=dict(
                num_layers=6,
                layer_cfg=dict(
                    self_attn_cfg=dict(embed_dims=256, num_levels=4, dropout=0.),
                    ffn_cfg=dict(embed_dims=256, feedforward_channels=2048, ffn_drop=0.),
                ),
            ),
            decoder=dict(
                num_layers=6,
                return_intermediate=True,
                layer_cfg=dict(
                    self_attn_cfg=dict(embed_dims=256, num_heads=8, dropout=0.),
                    cross_attn_cfg=dict(embed_dims=256, num_levels=4, dropout=0.),
                    ffn_cfg=dict(embed_dims=256, feedforward_channels=2048, ffn_drop=0.),
                ),
                post_norm_cfg=None,
            ),
        ),
        positional_encoding=dict(offset=-0.5, temperature=10000, num_feats=128),
        loss_cls=dict(type='QualityFocalLoss', use_sigmoid=True, beta=2.0, loss_weight=1.),
        loss_bbox=dict(type='L1Loss', loss_weight=5.),
        loss_iou=dict(type='GIoULoss', loss_weight=2.),
    ),
    rpn_head=dict(
        type='RPNHead',
        in_channels=256,
        feat_channels=256,
        anchor_generator=dict(
            type='AnchorGenerator',
            octave_base_scale=4,
            scales_per_octave=3,
            ratios=[0.5, 1.0, 2.0],
            strides=[4, 8, 16, 32, 64],
        ),
        bbox_coder=dict(
            type='DeltaXYWHBBoxCoder',
            target_means=[.0, .0, .0, .0],
            target_stds=[1.0, 1.0, 1.0, 1.0],
        ),
        loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=True, loss_weight=12.),
        loss_bbox=dict(type='L1Loss', loss_weight=12.),
    ),
    roi_head=[dict(
        type='StandardRoIHead',
        bbox_roi_extractor=dict(
            type='SingleRoIExtractor',
            roi_layer=dict(type='RoIAlign', output_size=7, sampling_ratio=0),
            out_channels=256,
            featmap_strides=[4, 8, 16, 32],
        ),
        bbox_head=dict(
            type='Shared2FCBBoxHead',
            in_channels=256,
            fc_out_channels=1024,
            roi_feat_size=7,
            num_classes=NUM_CLASSES,
            bbox_coder=dict(
                type='DeltaXYWHBBoxCoder',
                target_means=[0., 0., 0., 0.],
                target_stds=[0.1, 0.1, 0.2, 0.2],
            ),
            reg_class_agnostic=False,
            loss_cls=dict(type='CrossEntropyLoss', use_sigmoid=False, loss_weight=12.),
            loss_bbox=dict(type='L1Loss', loss_weight=12.),
        ),
    )],
    bbox_head=[dict(
        type='ATSSHead',
        num_classes=NUM_CLASSES,
        in_channels=256,
        stacked_convs=4,
        feat_channels=256,
        anchor_generator=dict(
            type='AnchorGenerator',
            ratios=[1.0],
            octave_base_scale=8,
            scales_per_octave=1,
            center_offset=0.0,
            strides=[4, 8, 16, 32, 64],
        ),
        bbox_coder=dict(
            type='DeltaXYWHBBoxCoder',
            target_means=[.0, .0, .0, .0],
            target_stds=[0.1, 0.1, 0.2, 0.2],
        ),
        loss_cls=dict(type='FocalLoss', use_sigmoid=True, gamma=2.0, alpha=0.25, loss_weight=12.),
        loss_bbox=dict(type='GIoULoss', loss_weight=24.),
        loss_centerness=dict(type='CrossEntropyLoss', use_sigmoid=True, loss_weight=12.),
    )],
    train_cfg=[
        dict(
            assigner=dict(
                type='HungarianAssigner',
                match_costs=[
                    dict(type='FocalLossCost', weight=2.0),
                    dict(type='BBoxL1Cost', weight=5.0, box_format='xywh'),
                    dict(type='IoUCost', iou_mode='giou', weight=2.0),
                ],
            ),
        ),
        dict(
            rpn=dict(
                assigner=dict(
                    type='MaxIoUAssigner',
                    pos_iou_thr=0.7, neg_iou_thr=0.3,
                    min_pos_iou=0.3, ignore_iof_thr=-1,
                ),
                sampler=dict(
                    type='RandomSampler',
                    num=256, pos_fraction=0.5,
                    neg_pos_ub=-1, add_gt_as_proposals=False,
                ),
                allowed_border=-1, pos_weight=-1, debug=False,
            ),
            rpn_proposal=dict(
                nms_pre=4000, max_per_img=1000,
                nms=dict(type='nms', iou_threshold=0.7), min_bbox_size=0,
            ),
            rcnn=dict(
                assigner=dict(
                    type='MaxIoUAssigner',
                    pos_iou_thr=0.5, neg_iou_thr=0.5,
                    min_pos_iou=0.5, ignore_iof_thr=-1,
                ),
                sampler=dict(
                    type='RandomSampler',
                    num=512, pos_fraction=0.25,
                    neg_pos_ub=-1, add_gt_as_proposals=True,
                ),
                pos_weight=-1, debug=False,
            ),
        ),
        dict(
            assigner=dict(type='ATSSAssigner', topk=9),
            allowed_border=-1, pos_weight=-1, debug=False,
        ),
    ],
    test_cfg=[
        dict(max_per_img=300),
        dict(
            rpn=dict(
                nms_pre=1000, max_per_img=1000,
                nms=dict(type='nms', iou_threshold=0.7), min_bbox_size=0,
            ),
            rcnn=dict(
                score_thr=0.0,
                nms=dict(type='nms', iou_threshold=0.5),
                max_per_img=100,
            ),
        ),
        dict(
            nms_pre=1000, min_bbox_size=0, score_thr=0.05,
            nms=dict(type='nms', iou_threshold=0.6), max_per_img=100,
        ),
    ],
)

print('Model config defined.')
print('  Backbone : SwinTransformer-Tiny (ImageNet pretrained)')
print('  Neck     : ChannelMapper (4 scales, 256 ch)')
print(f'  Head     : CoDINOHead (300 queries, {NUM_CLASSES} classes)')
print('  Aux      : RPNHead + Shared2FCBBoxHead + ATSSHead (train only)')

## Training

Uses MMEngine `Runner` for training. Key settings:
- **Optimizer**: AdamW lr=1e-4, backbone lr=1e-5 (10x lower)
- **Schedule**: Linear warmup (2000 iters) + MultiStep decay at epochs 27 & 33
- **Checkpointing**: saves best by `coco/bbox_mAP_50` on validation set

In [ ]:
from mmengine.config import Config
from mmengine.runner import Runner

cfg_dict = dict(
    model=model_cfg,

    # =========================
    # DATALOADERS
    # =========================
    train_dataloader=dict(
        batch_size=2,
        num_workers=2,
        persistent_workers=True,
        sampler=dict(type='DefaultSampler', shuffle=True),
        dataset=dict(
            type='TACODataset',
            data_root=DATASET_DIR,
            ann_file='train_annotations.json',
            data_prefix=dict(img='images/train/'),
            filter_cfg=dict(filter_empty_gt=True, min_size=32),
            pipeline=train_pipeline,
        ),
    ),
    val_dataloader=dict(
        batch_size=1,
        num_workers=2,
        persistent_workers=True,
        drop_last=False,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='TACODataset',
            data_root=DATASET_DIR,
            ann_file='val_annotations.json',
            data_prefix=dict(img='images/val/'),
            test_mode=True,
            pipeline=test_pipeline,
        ),
    ),
    test_dataloader=dict(
        batch_size=1,
        num_workers=2,
        persistent_workers=True,
        drop_last=False,
        sampler=dict(type='DefaultSampler', shuffle=False),
        dataset=dict(
            type='TACODataset',
            data_root=DATASET_DIR,
            ann_file='test_annotations.json',
            data_prefix=dict(img='images/test/'),
            test_mode=True,
            pipeline=test_pipeline,
        ),
    ),

    # =========================
    # EVALUATORS
    # =========================
    val_evaluator=dict(
        type='CocoMetric',
        ann_file=os.path.join(DATASET_DIR, 'val_annotations.json'),
        metric='bbox',
        format_only=False,
    ),
    test_evaluator=dict(
        type='CocoMetric',
        ann_file=os.path.join(DATASET_DIR, 'test_annotations.json'),
        metric='bbox',
        format_only=False,
    ),

    # =========================
    # TRAINING LOOPS
    # =========================
    train_cfg=dict(type='EpochBasedTrainLoop', max_epochs=36, val_interval=1),
    val_cfg=dict(type='ValLoop'),
    test_cfg=dict(type='TestLoop'),

    # =========================
    # OPTIMIZER
    # =========================
    optim_wrapper=dict(
        type='OptimWrapper',
        optimizer=dict(type='AdamW', lr=1e-4, weight_decay=1e-4),
        clip_grad=dict(max_norm=0.1, norm_type=2),
        paramwise_cfg=dict(
            custom_keys={
                'backbone': dict(lr_mult=0.1),
                'sampling_offsets': dict(lr_mult=0.1),
                'reference_points': dict(lr_mult=0.1),
            }
        ),
    ),

    # =========================
    # LR SCHEDULE
    # =========================
    param_scheduler=[
        dict(type='LinearLR', start_factor=1e-4, by_epoch=False, begin=0, end=2000),
        dict(type='MultiStepLR', begin=0, end=36, by_epoch=True,
             milestones=[27, 33], gamma=0.1),
    ],

    # =========================
    # HOOKS
    # =========================
    default_hooks=dict(
        timer=dict(type='IterTimerHook'),
        logger=dict(type='LoggerHook', interval=50),
        param_scheduler=dict(type='ParamSchedulerHook'),
        checkpoint=dict(
            type='CheckpointHook',
            interval=1,
            max_keep_ckpts=3,
            save_best='coco/bbox_mAP_50',
        ),
        sampler_seed=dict(type='DistSamplerSeedHook'),
        visualization=dict(type='DetVisualizationHook'),
    ),

    # =========================
    # ENVIRONMENT
    # =========================
    env_cfg=dict(
        cudnn_benchmark=False,
        mp_cfg=dict(mp_start_method='spawn', opencv_num_threads=0),
        dist_cfg=dict(backend='gloo'),
    ),
    vis_backends=[dict(type='LocalVisBackend')],
    visualizer=dict(
        type='DetLocalVisualizer',
        vis_backends=[dict(type='LocalVisBackend')],
        name='visualizer',
    ),
    log_processor=dict(type='LogProcessor', window_size=50, by_epoch=True),
    log_level='INFO',
    load_from=None,
    resume=False,
    work_dir=OUTPUT_DIR,
)

cfg    = Config(cfg_dict)
runner = Runner.from_cfg(cfg)
runner.train()

## Test Evaluation

Loads the best checkpoint, runs inference on the test set, and reports:
- COCO mAP@50 and mAP@50-95 (via pycocotools)
- Counting MAE, within ±1, within ±3
- Average inference time

In [ ]:
from mmdet.apis import init_detector, inference_detector

# Find best checkpoint saved by CheckpointHook
ckpt_list = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'best_coco_bbox_mAP_50_epoch_*.pth')))
if not ckpt_list:
    ckpt_list = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'epoch_*.pth')))
if not ckpt_list:
    raise FileNotFoundError(f'No checkpoints found in {OUTPUT_DIR}')

best_ckpt = ckpt_list[-1]
print(f'Loading: {best_ckpt}')

infer_model = init_detector(cfg, checkpoint=best_ckpt, device=DEVICE)
infer_model.eval()
print('Model ready for inference.')

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

# Load test annotations
with open(os.path.join(DATASET_DIR, 'test_annotations.json')) as f:
    test_coco_data = json.load(f)

gt_counts = defaultdict(int)
for ann in test_coco_data['annotations']:
    gt_counts[ann['image_id']] += 1

test_img_dir    = os.path.join(DATASET_DIR, 'images', 'test')
coco_preds      = []
count_errors    = []
inference_times = []
per_image_results = []

for img_info in tqdm(test_coco_data['images'], desc='Test inference'):
    img_path = os.path.join(test_img_dir, img_info['file_name'])
    if not os.path.exists(img_path):
        continue

    t0     = time.perf_counter()
    result = inference_detector(infer_model, img_path)
    inf_ms = (time.perf_counter() - t0) * 1000
    inference_times.append(inf_ms)

    pred   = result.pred_instances
    keep   = pred.scores >= CONF_THRESHOLD
    boxes  = pred.bboxes[keep].cpu().numpy()
    scores = pred.scores[keep].cpu().numpy()
    labels = pred.labels[keep].cpu().numpy()

    img_id    = img_info['id']
    gt_count  = gt_counts.get(img_id, 0)
    pred_count = len(boxes)
    count_errors.append(abs(pred_count - gt_count))

    per_image_results.append({
        'file':         img_info['file_name'],
        'gt_count':     gt_count,
        'pred_count':   pred_count,
        'count_error':  abs(pred_count - gt_count),
        'inference_ms': inf_ms,
    })

    # mmdet labels are 0-indexed; COCO category_id is 1-indexed
    img_h, img_w = cv2.imread(img_path).shape[:2]
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = box.tolist()
        coco_preds.append({
            'image_id':    img_id,
            'category_id': int(label) + 1,
            'bbox':        [x1, y1, x2 - x1, y2 - y1],
            'score':       float(score),
        })

# =========================
# COCO mAP
# =========================
gt_coco = COCO()
gt_coco.dataset = test_coco_data
gt_coco.createIndex()

box_map50 = box_map50_95 = 0.0
if coco_preds:
    dt_coco      = gt_coco.loadRes(coco_preds)
    ev           = COCOeval(gt_coco, dt_coco, 'bbox')
    ev.evaluate(); ev.accumulate(); ev.summarize()
    box_map50_95 = float(ev.stats[0])
    box_map50    = float(ev.stats[1])
else:
    print('No predictions above threshold.')

# =========================
# COUNTING METRICS
# =========================
within_1 = sum(1 for e in count_errors if e <= 1) / len(count_errors) * 100
within_3 = sum(1 for e in count_errors if e <= 3) / len(count_errors) * 100
avg_inf  = float(np.mean(inference_times[1:]) if len(inference_times) > 1 else inference_times[0])

print('\n' + '=' * 50)
print('CO-DINO TEST RESULTS')
print('=' * 50)
print(f'Box mAP@50:            {box_map50:.4f}')
print(f'Box mAP@50-95:         {box_map50_95:.4f}')
print(f'Counting MAE:          {np.mean(count_errors):.2f}')
print(f'Counting within +/-1:  {within_1:.1f}%')
print(f'Counting within +/-3:  {within_3:.1f}%')
print(f'Avg inference time:    {avg_inf:.1f} ms')
print(f'Median inference time: {np.median(inference_times):.1f} ms')
print('=' * 50)

## Save Results

In [ ]:
results_data = {
    'metrics': {
        'box_map50':          box_map50,
        'box_map50_95':       box_map50_95,
        'mask_map50':         0.0,
        'mask_map50_95':      0.0,
        'counting_mae':       float(np.mean(count_errors)),
        'counting_within_1':  within_1,
        'counting_within_3':  within_3,
        'avg_inference_ms':   avg_inf,
    },
    'per_image': per_image_results,
}

out_path = os.path.join(OUTPUT_DIR, 'co_dino_results.json')
with open(out_path, 'w') as f:
    json.dump(results_data, f, indent=2)
print(f'Results saved to: {out_path}')

## Visualize Predictions

In [ ]:
COLORS = plt.cm.get_cmap('tab10', NUM_CLASSES)

sample_infos = [
    i for i in test_coco_data['images']
    if os.path.exists(os.path.join(test_img_dir, i['file_name']))
][:8]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, img_info in zip(axes.flat, sample_infos):
    img_path = os.path.join(test_img_dir, img_info['file_name'])
    image    = cv2.imread(img_path)
    image    = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    result = inference_detector(infer_model, img_path)
    pred   = result.pred_instances
    keep   = pred.scores >= CONF_THRESHOLD
    boxes  = pred.bboxes[keep].cpu().numpy()
    scores = pred.scores[keep].cpu().numpy()
    labels = pred.labels[keep].cpu().numpy()

    overlay = image.copy()
    for box, score, label in zip(boxes, scores, labels):
        x1, y1, x2, y2 = map(int, box)
        color = tuple(int(c * 255) for c in COLORS(int(label))[:3])
        cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
        text = f'{CLASS_NAMES[label]} {score:.2f}'
        cv2.putText(overlay, text, (x1, max(y1 - 4, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

    gt_count = gt_counts.get(img_info['id'], 0)
    ax.imshow(overlay)
    ax.set_title(f'GT:{gt_count} Pred:{len(boxes)}', fontsize=9)
    ax.axis('off')

plt.suptitle('Co-DINO Predictions on Test Set (conf≥0.5)', fontsize=13)
plt.tight_layout()
viz_path = os.path.join(OUTPUT_DIR, 'co_dino_predictions.png')
plt.savefig(viz_path, dpi=150)
plt.show()
print(f'Saved: {viz_path}')